## Generates the synthetic Data for Training the Model

First we need to do our imports and get our master csv file.

In [1]:
import pandas as pd
import nvdlib
import pprint
import json
import os
from dotenv import load_dotenv
import urllib.parse
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing import Literal

# import playwright tool
import playwright_tool

In [2]:
master_df = pd.read_csv('normal_master.csv')
master_df.head()

,model_name,vendor,device_type,keyword_lookup
0,Xbox Series X,Microsoft,Gaming Console,Microsoft Xbox
1,Nintendo Switch,Nintendo,Gaming Console,Nintendo Switch
2,PlayStation 5,Sony,Gaming Console,Sony PlayStation
3,Plex Media Server,Plex,Media Server,Plex
4,Lockerstor (ADM),Asustor,NAS,Asustor


Now we connect to the NVD API.

In [3]:
load_dotenv()

nvd_api_key = os.getenv('NVD_API_KEY')

In [4]:
request = nvdlib.searchCVE(cveId='CVE-2021-26855', key=nvd_api_key, delay=2)
if request:
    pprint.pprint(request)

[{'id': 'CVE-2021-26855', 'sourceIdentifier': 'secure@microsoft.com', 'published': '2021-03-03T00:15:12.103', 'lastModified': '2025-10-30T19:55:58.670', 'vulnStatus': 'Analyzed', 'cveTags': [], 'descriptions': [{'lang': 'en', 'value': 'Microsoft Exchange Server Remote Code Execution Vulnerability'}, {'lang': 'es', 'value': 'Una Vulnerabilidad de Ejecución de código remota de Microsoft Exchange Server. Este ID de CVE es diferente de CVE-2021-26412, CVE-2021-26854, CVE-2021-26857, CVE-2021-26858, CVE-2021-27065, CVE-2021-27078'}], 'metrics': {'cvssMetricV31': [{'source': 'secure@microsoft.com', 'type': 'Secondary', 'cvssData': {'version': '3.1', 'vectorString': 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:N', 'baseScore': 9.1, 'baseSeverity': 'CRITICAL', 'attackVector': 'NETWORK', 'attackComplexity': 'LOW', 'privilegesRequired': 'NONE', 'userInteraction': 'NONE', 'scope': 'UNCHANGED', 'confidentialityImpact': 'HIGH', 'integrityImpact': 'HIGH', 'availabilityImpact': 'NONE'}, 'exploitabilit

In [5]:
device_name = "Ring Video Doorbell"

# Search for CVEs containing the device name
# limit=25 restricts the results to the top 25 matches
results = nvdlib.searchCVE(keywordSearch=device_name, limit=25, key=nvd_api_key, delay=2)

for entry in results:
    print(entry)

{'id': 'CVE-2015-4400', 'sourceIdentifier': 'cve@mitre.org', 'published': '2018-02-06T16:29:00.527', 'lastModified': '2024-11-21T02:31:00.143', 'vulnStatus': 'Modified', 'cveTags': [], 'descriptions': [{'lang': 'en', 'value': 'Ring (formerly DoorBot) video doorbells allow remote attackers to obtain sensitive information about the wireless network configuration by pressing the set up button and leveraging an API in the GainSpan Wi-Fi module.'}, {'lang': 'es', 'value': 'Los videoporteros Ring (anteriormente DoorBot) permiten que atacantes remotos obtengan información sensible sobre la configuración de red inalámbrica presionando el botón de configuración y utilizando una API en el módulo Wi-Fi GainSpan.'}], 'metrics': {'cvssMetricV30': [{'source': 'nvd@nist.gov', 'type': 'Primary', 'cvssData': {'version': '3.0', 'vectorString': 'CVSS:3.0/AV:P/AC:L/PR:N/UI:N/S:U/C:H/I:N/A:N', 'baseScore': 4.6, 'baseSeverity': 'MEDIUM', 'attackVector': 'PHYSICAL', 'attackComplexity': 'LOW', 'privilegesRequ

Now we will get all CVEs for the devices we are interested in and store them in a json file. Limit to top 25 CVEs returned by the API.

In [6]:
def extract_cve_context(nvd_data):
    """
    Parses a raw NVD API response to extract high-value context
    for LLM remediation analysis.
    """
    
    # 1. Extract ID and Description (English)
    cve_id = nvd_data.id
    
    # distinct descriptions often exist for different languages
    descriptions = nvd_data.descriptions
    description_text = next(
        (item.value for item in descriptions if item.lang == 'en'), 
        "No description available."
    )

    # this is a sanitization step
    if description_text:
        description_text = description_text.replace('\n', ' ').strip()

    refs = nvd_data.references
    reference_urls = list({r.url for r in refs})

    # Build the structured context object
    return {
        "cve_id": cve_id,
        "description": description_text,
        "reference_links": reference_urls
    }

In [7]:
# Helper function to serialize CVE object into a JSON object
def nvd_santizer(obj):
    if hasattr(obj, '__dict__'):
        obj["description"] = obj["description"].replace("\n", " ").strip()
        return obj.__dict__
    # Fallback: If it's something weird like a date object, convert to string
    print("FALLBACK TRIGGERED")
    return str(obj)

In [8]:
non_dup_keywords = master_df["keyword_lookup"].unique()

def create_cve_objects():
    # Open file in 'append' mode ('a') so we save progress as we go
    with open("discovered_cves.jsonl", "a", encoding="utf-8") as f:
        
        for keyword in non_dup_keywords:
            
            # Initialize the list for this specific keyword
            current_findings = []

            keyword_serialized = urllib.parse.quote_plus(keyword)

            # Run your search (Assuming you fixed the encoding/nvdlib parts)
            results = nvdlib.searchCVE(keywordSearch=keyword_serialized, limit=25, key=nvd_api_key, delay=1)
            
            for entry in results:
                # Extract your data (Assuming this function is now compatible)
                data = extract_cve_context(entry)
                current_findings.append(data)

            # 4. Save Strategy: Write ONLY this keyword's data to disk immediately
            # We wrap it in a mini-dict so we know which keyword these results belong to
            record_to_save = {keyword: current_findings}

            print(record_to_save)
            
            # Dump to JSON string and add a newline (required for JSONL)
            f.write(json.dumps(record_to_save, default=nvd_santizer) + "\n")
            
            # Force the OS to write to the hard drive NOW (prevents data loss if script crashes)
            f.flush() 

# create_cve_objects()
print("Scan complete. Data saved to discovered_cves.jsonl")

Scan complete. Data saved to discovered_cves.jsonl


Validate JSONL completeness and syntactical correctness.

In [9]:
with open('discovered_cves.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        try:
            record = json.loads(line)
            print(record)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            print(f"Offending line: {line}")

{'Microsoft Xbox': [{'cve_id': 'CVE-2007-1220', 'description': 'The Hypervisor in Microsoft Xbox 360 kernel 4532 and 4548 does not properly verify the parameters passed to the syscall dispatcher, which allows attackers with physical access to bypass code-signing requirements and execute arbitrary code.', 'reference_links': ['http://securityreason.com/securityalert/2367', 'http://www.securityfocus.com/archive/1/461489/100/0/threaded', 'http://www.securityfocus.com/bid/22745']}, {'cve_id': 'CVE-2007-1221', 'description': 'The Hypervisor in Microsoft Xbox 360 kernel 4532 and 4548 allows attackers with physical access to force execution of the hypervisor syscall with a certain register set, which bypasses intended code protection.', 'reference_links': ['http://www.securityfocus.com/archive/1/463974/100/200/threaded', 'http://securityreason.com/securityalert/2367', 'http://www.securityfocus.com/archive/1/461489/100/0/threaded', 'http://www.securityfocus.com/bid/22745']}, {'cve_id': 'CVE-202

Now we will define the SearXNG tool so the llm can use it if it needs to.

In [10]:
class SearXNGTool(BaseModel):
    """Tool to search the internet. Calling this tool will give you
    a markdown representation of 5 **full** websites."""
    query: str = Field(..., description="The search query to look up.")

In [11]:
@tool(args_schema=SearXNGTool)
async def searxng_search(query: str) -> str:
    """Use SearXNG to search the internet and return markdown content of top 5 websites."""
    search_results = await playwright_tool.search_and_parse_web(query=query)
    return search_results

Test the tool.

In [12]:
result = await searxng_search.ainvoke({"query": "What is the capital of France?"})
print(result)

<begin_page>
<source url=https://en.wikipedia.org/wiki/Paris />
## Paris
Acèh
Адыгэбзэ
Адыгабзэ
Afrikaans
Alemannisch
አማርኛ
Anarâškielâ
अंगिका
Ænglisc
العربية
Aragonés
ܐܪܡܝܐ
Արեւմտահայերէն
Armãneashti
Arpetan
অসমীয়া
Asturianu
Atikamekw
अवधी
Avañe'ẽ
Авар
Aymar aru
Azərbaycanca
تۆرکجه
Basa Bali
Bamanankan
বাংলা
閩南語 / Bân-lâm-gí
Башҡортса
Беларуская
Беларуская (тарашкевіца)
भोजपुरी
Bikol Central
Bislama
Български
Boarisch
བོད་ཡིག
Bosanski
Brezhoneg
Буряад
Català
Чӑвашла
Cebuano
Čeština
Chamoru
Chavacano de Zamboanga
Chi-Chewa
ChiShona
ChiTumbuka
Corsu
Cymraeg
Dagbanli
Dansk
الدارجة
Davvisámegiella
Deitsch
Deutsch
Diné bizaad
Dolnoserbski
डोटेली
Eesti
Ελληνικά
Emiliàn e rumagnòl
Эрзянь
Español
Esperanto
Estremeñu
Euskara
Eʋegbe
فارسی
Fiji Hindi
Føroyskt
Français
Frysk
Fulfulde
Furlan
Gaeilge
Gaelg
Gagauz
Gàidhlig
Galego
ГӀалгӀай
贛語
گیلکی
ગુજરાતી
𐌲𐌿𐍄𐌹𐍃𐌺
गोंयची कोंकणी / Gõychi Konknni
Gungbe
客家語 / Hak-kâ-ngî
한국어
Hausa
Hawaiʻi
Հայերեն
हिन्दी
Hornjoserbsce
Hrvatski
Ido
Ilokano
Bahasa Indonesia